In [7]:
import os
import pandas as pd
from pathlib import Path

# ============================================================
# CONFIGURATION - Update these paths to match your setup
# ============================================================
INPUT_DIR = "/Users/yashaswiaryan/Projects/The WOLF/The-Wolf/full_history"    # Folder with daily CSVs
OUTPUT_DIR = "data/stock_weekly/"          # Folder for weekly CSVs

# ============================================================
# TARGET STOCKS (subset for initial testing)
# ============================================================
# Apple, Amazon, Microsoft, Kroger Co, Alcoa Corp, Medtronic
TARGET_TICKERS = ["AAPL", "AMZN", "MSFT", "KR", "AA", "MDT"]

# ============================================================
# AGGREGATION RULES (Standard OHLCV weekly conversion)
# ============================================================
# Open   → First open of the week (how the week started)
# High   → Max high of the week   (peak price reached)
# Low    → Min low of the week    (lowest price reached)
# Close  → Last close of the week (how the week ended)
# Adj Close → Last adj close      (adjusted closing price)
# Volume → Sum of the week        (total shares traded)
# ============================================================

AGGREGATION_RULES = {
    'open':      'first',
    'high':      'max',
    'low':       'min',
    'close':     'last',
    'adj close': 'last',
    'volume':    'sum'
}


def convert_daily_to_weekly(filepath):
    """
    Reads a daily stock CSV and returns a weekly-aggregated DataFrame.
    
    Parameters:
        filepath (str): Path to the daily CSV file
        
    Returns:
        pd.DataFrame: Weekly aggregated stock data
    """
    # --- Step 1: Read the CSV ---
    df = pd.read_csv(filepath)
    
    # --- Step 2: Standardize column names (lowercase, strip spaces) ---
    df.columns = df.columns.str.strip().str.lower()
    
    # --- Step 3: Parse dates and sort ascending ---
    # (Your data is in descending order, so we sort ascending for correct aggregation)
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)
    
    # --- Step 4: Remove rows with missing critical data ---
    df = df.dropna(subset=['date', 'close'])
    
    # --- Step 5: Set date as index for resampling ---
    df = df.set_index('date')
    
    # --- Step 6: Resample to weekly frequency (W-FRI = week ending Friday) ---
    # This groups all trading days Mon-Fri into one weekly row
    weekly = df.resample('W-FRI').agg(AGGREGATION_RULES)
    
    # --- Step 7: Drop weeks with no trading data ---
    # (e.g., holiday weeks where market was closed entirely)
    weekly = weekly.dropna(subset=['close'])
    
    # --- Step 8: Round for cleaner output ---
    weekly = weekly.round(4)
    
    # --- Step 9: Reset index so 'date' becomes a column again ---
    weekly = weekly.reset_index()
    
    return weekly


def process_all_stocks(input_dir, output_dir):
    """
    Processes all daily stock CSVs in input_dir and saves weekly versions to output_dir.
    
    Parameters:
        input_dir  (str): Folder containing daily stock CSVs (e.g., AAPL.csv)
        output_dir (str): Folder to save weekly aggregated CSVs
    """
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Get only the target stock CSV files
    csv_files = []
    for ticker in TARGET_TICKERS:
        fp = Path(input_dir) / f"{ticker}.csv"
        if fp.exists():
            csv_files.append(fp)
        else:
            print(f"  ⚠️  File not found: {fp}")
    
    if not csv_files:
        print(f"⚠️  No matching CSV files found in: {input_dir}")
        return
    
    print(f"📂 Found {len(csv_files)} stock files to process\n")
    
    success_count = 0
    error_files = []
    summary_rows = []
    
    for filepath in csv_files:
        ticker = filepath.stem  # e.g., "AAPL" from "AAPL.csv"
        
        try:
            weekly_df = convert_daily_to_weekly(filepath)
            
            # Save to output directory
            output_path = os.path.join(output_dir, f"{ticker}.csv")
            weekly_df.to_csv(output_path, index=False)
            
            # Track stats
            daily_count = len(pd.read_csv(filepath))
            weekly_count = len(weekly_df)
            date_range = f"{weekly_df['date'].min().date()} → {weekly_df['date'].max().date()}"
            
            summary_rows.append({
                'ticker': ticker,
                'daily_rows': daily_count,
                'weekly_rows': weekly_count,
                'date_range': date_range
            })
            
            success_count += 1
            print(f"  ✅ {ticker:6s} | {daily_count:>6} daily → {weekly_count:>5} weekly | {date_range}")
            
        except Exception as e:
            error_files.append((ticker, str(e)))
            print(f"  ❌ {ticker:6s} | Error: {e}")
    
    # --- Final Summary ---
    print(f"\n{'='*60}")
    print(f"📊 Processing Complete!")
    print(f"   ✅ Success: {success_count}/{len(csv_files)}")
    if error_files:
        print(f"   ❌ Errors:  {len(error_files)}")
        for ticker, err in error_files:
            print(f"      - {ticker}: {err}")
    print(f"   📁 Output:  {output_dir}")
    print(f"{'='*60}")
    
    # Optional: Save summary as CSV
    if summary_rows:
        summary_df = pd.DataFrame(summary_rows)
        summary_path = os.path.join(output_dir, "_processing_summary.csv")
        summary_df.to_csv(summary_path, index=False)
        print(f"\n📋 Summary saved to: {summary_path}")


# ============================================================
# RUN
# ============================================================
if __name__ == "__main__":
    process_all_stocks(INPUT_DIR, OUTPUT_DIR)

📂 Found 6 stock files to process

  ✅ AAPL   |  10852 daily →  2247 weekly | 1980-12-12 → 2023-12-29
  ✅ AMZN   |   6700 daily →  1390 weekly | 1997-05-16 → 2023-12-29
  ✅ MSFT   |   9526 daily →  1973 weekly | 1986-03-14 → 2023-12-29
  ✅ KR     |  10907 daily →  2257 weekly | 1977-01-07 → 2020-04-03
  ✅ AA     |  15605 daily →  3235 weekly | 1962-01-05 → 2023-12-29
  ✅ MDT    |  11899 daily →  2462 weekly | 1973-05-04 → 2020-07-03

📊 Processing Complete!
   ✅ Success: 6/6
   📁 Output:  data/stock_weekly/

📋 Summary saved to: data/stock_weekly/_processing_summary.csv


In [13]:
news_appl.head()

,Unnamed: 0,Date,Article_title,Stock_symbol,Url,Publisher,Author,Article,Lsa_summary,Luhn_summary,Textrank_summary,Lexrank_summary
0,NaN,2020-06-10 07:33:26 UTC,Tech Stocks And FAANGS Strong Again To Start D...,AAPL,https://www.benzinga.com/government/20/06/1622...,JJ Kinahan,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,2020-06-10 04:14:08 UTC,10 Biggest Price Target Changes For Wednesday,AAPL,https://www.benzinga.com/analyst-ratings/price...,Lisa Levin,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,2020-06-10 03:53:47 UTC,"Benzinga Pro's Top 5 Stocks To Watch For Wed.,...",AAPL,https://www.benzinga.com/short-sellers/20/06/1...,Benzinga Newsdesk,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,2020-06-10 03:19:25 UTC,"Deutsche Bank Maintains Buy on Apple, Raises P...",AAPL,https://www.benzinga.com/news/20/06/16219873/d...,Benzinga Newsdesk,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,2020-06-10 02:27:11 UTC,Apple To Let Users Trade In Their Mac Computer...,AAPL,https://www.benzinga.com/news/20/06/16218697/a...,Neer Varshney,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
news_appl= pd.read_csv('/Users/yashaswiaryan/Projects/The WOLF/The-Wolf/processed_stock_news_data/AAPL.csv')
news_appl.columns

Index(['Unnamed: 0', 'Date', 'Article_title', 'Stock_symbol', 'Url',
       'Publisher', 'Author', 'Article', 'Lsa_summary', 'Luhn_summary',
       'Textrank_summary', 'Lexrank_summary'],
      dtype='object')

In [12]:
# ============================================================
# CONFIGURATION
# ============================================================
NEWS_DIR = "/Users/yashaswiaryan/Projects/The WOLF/The-Wolf/processed_stock_news_data"  # Update this to your news folder path
TARGET_TICKERS = ["AAPL", "AMZN", "MSFT", "KR", "AA", "MDT"]

# FinBERT token limit (approx word equivalent)
FINBERT_WORD_LIMIT = 400  # ~512 tokens ≈ 380-400 words

# ============================================================
# EXPLORATION
# ============================================================

summary_cols = ['Lsa_summary', 'Luhn_summary', 'Textrank_summary', 'Lexrank_summary']

for ticker in TARGET_TICKERS:
    filepath = Path(NEWS_DIR) / f"{ticker}.csv"
    
    if not filepath.exists():
        print(f"⚠️  File not found: {filepath}")
        continue
    
    df = pd.read_csv(filepath)
    
    print(f"\n{'='*70}")
    print(f"📰 {ticker} — {len(df)} articles")
    print(f"{'='*70}")
    
    # --- 1. Date range ---
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    print(f"\n📅 Date range: {df['Date'].min()} → {df['Date'].max()}")
    print(f"   Missing dates: {df['Date'].isna().sum()}")
    
    # --- 2. Null check for all important columns ---
    print(f"\n🔍 Null counts:")
    for col in ['Article_title', 'Article'] + summary_cols:
        if col in df.columns:
            null_count = df[col].isna().sum()
            null_pct = 100 * null_count / len(df)
            print(f"   {col:25s} → {null_count:>5} nulls ({null_pct:.1f}%)")
    
    # --- 3. Empty string check (not null but empty) ---
    print(f"\n🔍 Empty string counts:")
    for col in ['Article'] + summary_cols:
        if col in df.columns:
            empty_count = (df[col].astype(str).str.strip() == '').sum()
            print(f"   {col:25s} → {empty_count:>5} empty")
    
    # --- 4. Word count stats for Article and each summary ---
    print(f"\n📏 Word count statistics:")
    for col in ['Article'] + summary_cols:
        if col in df.columns:
            word_counts = df[col].astype(str).str.split().str.len()
            exceeds = (word_counts > FINBERT_WORD_LIMIT).sum()
            exceeds_pct = 100 * exceeds / len(df)
            print(f"\n   {col}:")
            print(f"     Mean: {word_counts.mean():.0f} | Median: {word_counts.median():.0f} | Max: {word_counts.max():.0f}")
            print(f"     Exceeds {FINBERT_WORD_LIMIT} words: {exceeds} articles ({exceeds_pct:.1f}%)")
    
    # --- 5. Duplicate check ---
    dup_titles = df['Article_title'].duplicated().sum()
    dup_articles = df['Article'].duplicated().sum()
    print(f"\n🔁 Duplicates:")
    print(f"   Duplicate titles:   {dup_titles}")
    print(f"   Duplicate articles: {dup_articles}")
    
    # --- 6. Check for HTML tags in articles ---
    html_pattern = r'<[^>]+>'
    for col in ['Article'] + summary_cols:
        if col in df.columns:
            html_count = df[col].astype(str).str.contains(html_pattern, regex=True).sum()
            if html_count > 0:
                print(f"\n   ⚠️  HTML tags found in {col}: {html_count} rows")

    # --- 7. Sample of a summary for visual inspection ---
    print(f"\n📝 Sample Lexrank summary (first non-null):")
    sample = df['Lexrank_summary'].dropna().iloc[0] if 'Lexrank_summary' in df.columns else "N/A"
    print(f"   {str(sample)[:300]}...")

print(f"\n\n{'='*70}")
print("✅ Exploration complete! Use these results to decide:")
print("   1. Which summary column has fewest nulls?")
print("   2. Which summary fits within FinBERT's 400-word limit?")
print("   3. How many duplicates and HTML issues need cleaning?")
print(f"{'='*70}")


📰 AAPL — 9338 articles

📅 Date range: 2020-03-09 00:00:00+00:00 → 2023-12-16 22:00:00+00:00
   Missing dates: 0

🔍 Null counts:
   Article_title             →     0 nulls (0.0%)
   Article                   →   473 nulls (5.1%)
   Lsa_summary               →   473 nulls (5.1%)
   Luhn_summary              →   473 nulls (5.1%)
   Textrank_summary          →   473 nulls (5.1%)
   Lexrank_summary           →   473 nulls (5.1%)

🔍 Empty string counts:
   Article                   →     0 empty
   Lsa_summary               →     0 empty
   Luhn_summary              →     0 empty
   Textrank_summary          →     0 empty
   Lexrank_summary           →     0 empty

📏 Word count statistics:

   Article:
     Mean: 730 | Median: 687 | Max: 13227
     Exceeds 400 words: 6957 articles (74.5%)

   Lsa_summary:
     Mean: 87 | Median: 90 | Max: 314
     Exceeds 400 words: 0 articles (0.0%)

   Luhn_summary:
     Mean: 84 | Median: 84 | Max: 312
     Exceeds 400 words: 0 articles (0.0%)

   Textra

IndexError: single positional indexer is out-of-bounds